# Analisis Eksperimen B2 — Naive RAG (Dense FAISS + Fixed Chunk + Top-4)

- **Konfigurasi:** `B2` (Base model sama dengan B0–B1, ditambah *dense retrieval* `intfloat/multilingual-e5-small` + `fixed` chunk + `top_k=4`)
- **Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` | **Seed:** `42` | **Split:** `dev.jsonl` (30 query)
- **Pertanyaan Riset (`NLP_IR.pdf`):** Apa efek penambahan retrieval dasar (*Naive RAG*) dibanding B1, dan penyebab error apa saja (retrieval vs context distraction vs token budget) yang muncul untuk memotivasi B3 & B4?

In [ ]:
import json, statistics
from pathlib import Path
from collections import Counter

ROOT = Path('..') if (Path('..') / 'runs').exists() else Path('.')
s_b0 = json.loads((ROOT / 'runs/B0/summary.json').read_text(encoding='utf-8'))
s_b1 = json.loads((ROOT / 'runs/B1/summary.json').read_text(encoding='utf-8'))
s_b2 = json.loads((ROOT / 'runs/B2/summary.json').read_text(encoding='utf-8'))
rows_b1 = {r['query_id']: r for r in (json.loads(l) for l in (ROOT / 'runs/B1/dev_outputs.jsonl').open(encoding='utf-8') if l.strip())}
rows = [json.loads(l) for l in (ROOT / 'runs/B2/dev_outputs.jsonl').open(encoding='utf-8') if l.strip()]

print('=== PERBANDINGAN PROGRES B0 -> B1 -> B2 ===')
print(f"{'Metrik':20s} | {'B0':>7s} | {'B1':>7s} | {'B2':>7s} | {'Delta B2-B1':>11s}")
print('-' * 64)
for k in ['json_valid', 'abstain_acc_unans', 'false_refusal_ans', 'citation_rate_ans', 'avg_latency_s']:
    d = s_b2[k] - s_b1[k]
    print(f'{k:20s} | {s_b0[k]:7.3f} | {s_b1[k]:7.3f} | {s_b2[k]:7.3f} | {d:+11.3f}')

In [ ]:
# Breakdown per Kategori + Penggunaan Token & Truncation
by_cat = {}
for r in rows:
    by_cat.setdefault(r['category'], []).append(r)

print(f"{'Kategori':18s} | {'N':>2s} | {'JSON':>4s} | {'Trunc':>5s} | {'Sup':>3s} | {'Unk':>3s} | {'AvgIn':>6s} | {'AvgOut':>6s} | {'Lat':>6s}")
print('-' * 82)
for cat, items in by_cat.items():
    jv = sum(1 for x in items if x['parsed_json'] is not None)
    tr = sum(1 for x in items if x['out_tokens'] >= 319)
    sup = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'supported')
    unk = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'unknown')
    ai = sum(x['in_tokens'] for x in items) / len(items)
    ao = sum(x['out_tokens'] for x in items) / len(items)
    al = sum(x['latency_s'] for x in items) / len(items)
    print(f'{cat:18s} | {len(items):2d} | {jv:4d} | {tr:5d} | {sup:3d} | {unk:3d} | {ai:6.0f} | {ao:6.1f} | {al:5.2f}s')

In [ ]:
# Audit Error 1 (Retrieval Diagnosis): Komposisi Sumber Terambil (KHI vs PA vs ART)
# Mengukur apakah Dense + Fixed berhasil menemukan KHI untuk soal pasal_definisi & multi_hop
doc_prefix_counts = Counter()
khi_hit_pasal_mh = 0
total_pasal_mh = 0

for r in rows:
    docs = r.get('retrieved_doc_ids', [])
    for d in docs:
        doc_prefix_counts[d.split('_')[0]] += 1
    if r['category'] in ('pasal_definisi', 'multi_hop'):
        total_pasal_mh += 1
        has_khi = any(d.startswith('KHI_') for d in docs)
        if has_khi:
            khi_hit_pasal_mh += 1
        print(f"{r['query_id']} [{r['category']:14s}] KHI_hit={str(has_khi):5s} | top4={docs}")

print(f'\nTotal Distribusi Sumber di Top-4 (30x4=120 slots): {dict(doc_prefix_counts)}')
print(f'KHI Hit Rate pada pasal_definisi + multi_hop: {khi_hit_pasal_mh}/{total_pasal_mh} ({khi_hit_pasal_mh/total_pasal_mh*100:.1f}%)')

In [ ]:
# Audit Error 2: Context Distraction pada Query Unanswerable (Bandingkan B1 vs B2)
print('=== UNANSWERABLE: B1 (tanpa konteks) vs B2 (dengan Naive RAG) ===')
for r in rows:
    if r['answerability'] == 'unanswerable':
        v1 = (rows_b1[r['query_id']]['parsed_json'] or {}).get('verdict')
        v2 = (r['parsed_json'] or {}).get('verdict')
        print(f"{r['query_id']} | B1={str(v1):9s} -> B2={str(v2):9s} | ret={r['retrieved_doc_ids']}")
        print('   Q:', r['query'])
        print('   B2 ans:', str((r['parsed_json'] or {}).get('answer', r['raw_output']))[:200])
        print()

In [ ]:
# Audit Error 3: Kasus Gagal JSON Parse & Truncation (out_tokens >= 319)
bad = [r for r in rows if r['parsed_json'] is None]
print(f'JSON gagal parse di B2: {len(bad)}/{len(rows)}')
for r in bad:
    print(f"  - {r['query_id']} [{r['category']}] in_tok={r['in_tokens']} out_tok={r['out_tokens']} tail={repr(r['raw_output'][-80:])}")

In [ ]:
# Visualisasi Perbandingan B0 vs B1 vs B2
try:
    import matplotlib.pyplot as plt
    metrics = ['json_valid', 'abstain_acc_unans', 'false_refusal_ans', 'citation_rate_ans']
    x = range(len(metrics))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([i - w for i in x], [s_b0[m] for m in metrics], width=w, label='B0 Direct')
    ax.bar(x, [s_b1[m] for m in metrics], width=w, label='B1 Structured')
    ax.bar([i + w for i in x], [s_b2[m] for m in metrics], width=w, label='B2 Naive RAG')
    ax.set_xticks(list(x))
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.05)
    ax.set_title('Perbandingan Metrik B0 vs B1 vs B2 pada Dev Set (n=30)')
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib belum terpasang; lewati grafik.')

### Kesimpulan Analisis B2 (Naive RAG) & Implikasi ke B3–B4
1. **Manfaat Nyata Retrieval Dasar:**
   - `false_refusal_ans` turun drastis dari **12,5% (B1) → 4,2% (B2)** (hanya 1/24 query *answerable* yang ditolak).
   - Jawaban menjadi jauh lebih kontekstual: pada `Q_016`, B2 tidak lagi mengarang `UHHS` (B0) atau `KUHPerdata 3/5` (B1), melainkan mengenali `Pasal 182 KHI` & `An-Nisa 176`. Pada `Q_056`, B2 mengambil putusan nyata (`PA_0219`, `PA_0240`).
2. **3 Akar Masalah Naive RAG yang Ditemukan (Sangat Penting untuk Laporan):**
   - **Retrieval Drowning (KHI tertenggelam):** Pada 13 query `pasal_definisi` + `multi_hop`, dokumen kanonik `KHI_*` hanya muncul di top-4 sebanyak **1/13 kali (7,7%)** karena corpus didominasi 250 putusan + 50 artikel dan *dense retrieval* saja lemah pada nomor pasal persis (sejalan dengan A2: Dense R@5 = 0.3958 vs BM25 R@5 = 0.8750). **→ Bukti kuat perlunya B3 (Hybrid BM25 + Dense + Heading Chunk + Reranker + metadata filter).**
   - **Context Distraction pada Query Unanswerable:** Pada `Q_082` (waris adat Batak Toba), B1 berhasil *abstain*, tetapi B2 justru terdistraksi oleh 4 chunk tidak relevan dan berhalusinasi dengan `verdict="supported"`. **→ Bukti kuat perlunya B4 (Evidence Sufficiency Check + Independent Claim Verifier).**
   - **Token Overhead & Truncation:** Konteks RAG menaikkan `in_tokens` (~546 → ~1.485) dan latensi (8,23s → 11,97s). Seluruh **5/30 kasus `bad_json` di B2** terjadi karena output terpotong pada `max_new_tokens=320` (ditambah wrapper ```` ```json ````). **→ Untuk B3+, naikkan `max_new_tokens=512` dan evaluasi di Ablation A7.**